# 10. Benchmark thống nhất và lựa chọn mô hình

Notebook này thay thế cách tổng hợp cũ vốn trộn kết quả 5-Fold của các mô hình cổ điển với một lần chia 80/20 của DistilBERT và có một số giá trị được nhập cứng. Quy trình mới dùng **Repeated Stratified Holdout** với cùng train/test indices cho tất cả mô hình, sau đó báo cáo trung bình và độ lệch chuẩn.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from common import set_global_seed
SEED = 42
set_global_seed(SEED)
print(f"Đã thiết lập seed: {SEED}")


## Nguyên tắc đánh giá

- Cùng seed và cùng test set cho năm mô hình trong từng lần chia.
- TF-IDF, TruncatedSVD và vocabulary chỉ fit trên train.
- BiLSTM và DistilBERT được khởi tạo lại ở mỗi split.
- Không nhập cứng Accuracy, Macro-F1, QWK hoặc thời gian huấn luyện.
- Kết quả cuối cùng dùng mean ± std qua các seed.

## Smoke test cho nhánh học máy cổ điển

Smoke test chỉ kiểm tra pipeline chạy đúng và lưu file với tiền tố `smoke_test_`. Không sử dụng số liệu này làm kết quả cuối của báo cáo.

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    import subprocess
    subprocess.run(
        [
            sys.executable,
            'src/run_unified_benchmark.py',
            '--quick',
            '--models', 'logreg', 'svm', 'xgboost',
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Đặt RUN_SMOKE_TEST=True để kiểm tra nhanh pipeline classical.')


## Benchmark đầy đủ trên ba lần chia

Chạy đủ BiLSTM và DistilBERT trên CPU có thể mất nhiều giờ. Trước khi chạy, nên hoàn thành tuning DistilBERT trong notebook 09 để tạo `best_distilbert_config.json`.

In [ ]:
RUN_FULL_BENCHMARK = False

if RUN_FULL_BENCHMARK:
    import subprocess
    subprocess.run(
        [
            sys.executable,
            'src/run_unified_benchmark.py',
            '--seeds', '42', '52', '62',
            '--models', 'logreg', 'svm', 'xgboost', 'bilstm', 'distilbert',
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Đặt RUN_FULL_BENCHMARK=True để chạy benchmark cuối cùng.')


## Đọc kết quả cuối

File `unified_repeated_holdout_summary.csv` chỉ xuất hiện sau khi chạy benchmark đầy đủ. Mỗi dòng gồm mean, std, min, max và số split của từng metric.

In [ ]:
from pathlib import Path
import pandas as pd

report_dir = Path('../outputs/reports')
summary_path = report_dir / 'unified_repeated_holdout_summary.csv'
metrics_path = report_dir / 'unified_repeated_holdout_metrics.csv'

if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
    print('\nBảng mean:')
    display(summary.pivot(index='model', columns='metric', values='mean').round(4))
    print('\nBảng std:')
    display(summary.pivot(index='model', columns='metric', values='std').round(4))
else:
    print('Chưa có benchmark đầy đủ. Không sử dụng file smoke_test làm kết quả báo cáo cuối.')

if metrics_path.exists():
    display(pd.read_csv(metrics_path))


## Phân tích hiệu năng theo độ dài văn bản

Sau khi có prediction file đầy đủ, chạy lại data quality/fairness audit để tạo lát cắt hiệu năng theo độ dài. Đây không phải fairness nhân khẩu học.

In [ ]:
RUN_SLICE_ANALYSIS = False

if RUN_SLICE_ANALYSIS:
    import subprocess
    subprocess.run(
        [sys.executable, 'src/data_quality_fairness.py'],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Đặt RUN_SLICE_ANALYSIS=True sau khi benchmark đầy đủ hoàn thành.')
